<a href="https://colab.research.google.com/github/Suleymanabdy/Data-Science-Checkpoints./blob/main/Pharmabot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# Installing the required packages
!pip install streamlit
!pip install pyngrok
!pip install streamlit-option-menu


In [5]:
%%writefile app.py

import json
from datetime import datetime
from typing import Dict, Any, Tuple

import streamlit as st

st.set_page_config(page_title="PHARMACARE", page_icon="💊", layout="centered")


KB: Dict[str, Dict[str, Any]] = {
    "common cold": {
        "aliases": ["cold", "common cold", "runny nose", "sneezing", "nasal congestion"],
        "overview": "Viral upper respiratory infection with runny/stuffy nose, sneezing, mild sore throat, mild cough.",
        "otc": [
            "Paracetamol/acetaminophen for fever/pain (avoid overdosing; check combination products)",
            "Nasal saline rinse/spray",
            "Short-term decongestant (e.g., pseudoephedrine/phenylephrine) — avoid with hypertension/heart issues",
            "Cough lozenges or honey (not for children <1 year)"
        ],
        "seek_care": [
            "High fever ≥ 38.5°C for >3 days",
            "Shortness of breath, chest pain, confusion",
            "Severe sore throat with pus, rash, or neck stiffness",
            "Symptoms persist >10 days or worsen"
        ],
        "notes": "Antibiotics are **not** useful for common cold. Rest and fluids help."
    },
    "flu": {
        "aliases": ["influenza", "flu", "body aches", "fever with chills"],
        "overview": "Sudden fever, chills, body aches, headache, cough, sore throat.",
        "otc": [
            "Paracetamol/acetaminophen for fever/pain",
            "Fluids, rest",
            "Consider clinician evaluation within 48 hours for possible antivirals if high-risk"
        ],
        "seek_care": [
            "Breathing difficulty, chest pain, confusion",
            "Persistent high fever",
            "Dehydration or inability to drink",
            "High-risk groups (pregnancy, elderly, chronic illness)"
        ],
        "notes": "Antibiotics usually **not** indicated."
    },
    "allergic rhinitis": {
        "aliases": ["allergy", "hay fever", "allergic rhinitis", "sneezing with itch", "itchy eyes"],
        "overview": "Allergic runny/stuffy nose with sneezing and itchy/watery eyes.",
        "otc": [
            "Non-drowsy antihistamine (e.g., cetirizine, loratadine)",
            "Intranasal steroid spray (e.g., fluticasone) used as directed",
            "Saline rinses"
        ],
        "seek_care": [
            "Severe uncontrolled symptoms despite OTC",
            "Wheezing/breathing difficulty (possible asthma)"
        ],
        "notes": "Avoid known triggers when possible."
    },
    "diarrhea (non-bloody)": {
        "aliases": ["diarrhea", "loose stools", "diarrhoea"],
        "overview": "Loose/watery stools without blood and without high fever.",
        "otc": [
            "Oral Rehydration Salts (ORS)",
            "Zinc supplements for 10–14 days (commonly used in children)",
            "Loperamide for adults **only if no fever or blood/mucus**"
        ],
        "seek_care": [
            "Blood in stool, high fever, severe abdominal pain",
            "Signs of dehydration (very thirsty, little urine, dizziness)",
            "Diarrhea >3 days or in very young/older adults"
        ],
        "notes": "Hydration is key. Avoid dairy and greasy foods temporarily."
    },
    "heartburn/acid reflux": {
        "aliases": ["heartburn", "acid reflux", "indigestion", "GERD"],
        "overview": "Burning chest/upper stomach discomfort after meals or lying down.",
        "otc": [
            "Antacids (calcium carbonate, magnesium/aluminum hydroxide)",
            "H2 blockers (e.g., famotidine)",
            "Short course of OTC PPI (e.g., omeprazole) if persistent"
        ],
        "seek_care": [
            "Chest pain with sweating, radiation to arm/jaw, or breathlessness",
            "Swallowing difficulty, vomiting blood, black stools",
            "Unintentional weight loss"
        ],
        "notes": "Avoid late heavy meals, alcohol, caffeine; elevate head of bed."
    },
    "headache (tension)": {
        "aliases": ["headache", "tension headache"],
        "overview": "Mild to moderate band-like head pain, often stress-related.",
        "otc": [
            "Paracetamol/acetaminophen",
            "Ibuprofen/NSAIDs if no contraindications (ulcer, kidney disease, pregnancy in 3rd trimester)"
        ],
        "seek_care": [
            "Sudden severe 'worst-ever' headache",
            "Neurologic signs (weakness, vision/speech changes)",
            "Headache after head injury",
            "Fever with neck stiffness"
        ],
        "notes": "Hydration, rest, stress reduction can help."
    },
    "sore throat (mild)": {
        "aliases": ["sore throat", "throat pain"],
        "overview": "Mild throat pain without red flags.",
        "otc": [
            "Paracetamol/acetaminophen",
            "Throat lozenges/sprays",
            "Warm fluids, honey/lemon (not for <1 year)"
        ],
        "seek_care": [
            "Severe pain, pus on tonsils, high fever",
            "Rash, neck stiffness, breathing difficulty"
        ],
        "notes": "Most are viral; antibiotics often not needed."
    },
    "skin rash (mild)": {
        "aliases": ["rash", "skin irritation", "mild dermatitis"],
        "overview": "Mild itchy rash without systemic symptoms.",
        "otc": [
            "Topical hydrocortisone 1% (short-term)",
            "Oral non-drowsy antihistamine for itch",
            "Unscented moisturizers"
        ],
        "seek_care": [
            "Widespread rash, fever, pain, infection signs",
            "Involvement of eyes/genitals, or blistering/peeling"
        ],
        "notes": "Avoid known irritants; patch test new products."
    },
    # Triage-first topics where self-medication is **not** recommended
    "possible malaria": {
        "aliases": ["malaria", "high fever travel", "fever with chills"],
        "overview": "Fever with chills/sweats, headache, fatigue; malaria is endemic in many regions.",
        "otc": [],
        "seek_care": [
            "Get a **rapid diagnostic test (RDT)** or lab testing immediately.",
            "Do **not** self-start antimalarials without testing and clinical guidance."
        ],
        "notes": "Urgent evaluation recommended."
    },
    "possible uti": {
        "aliases": ["uti", "painful urination", "burning urine"],
        "overview": "Painful/frequent urination; may need urine testing.",
        "otc": [
            "Phenazopyridine can help symptoms short-term (stains urine orange); **not a cure**"
        ],
        "seek_care": [
            "Fever, flank/back pain (possible kidney infection)",
            "Pregnancy, recurrent UTIs, male patients — need clinician review"
        ],
        "notes": "Antibiotics require clinician evaluation."
    },
}

# Map user free-text to a KB key

def normalize_condition(user_text: str) -> Tuple[str, str]:
    t = (user_text or "").strip().lower()
    if not t:
        return "", ""
    for key, data in KB.items():
        for alias in data["aliases"]:
            if alias in t:
                return key, alias
    # simple fallbacks
    if "fever" in t and ("chills" in t or "sweat" in t):
        return "possible malaria", "fever with chills"
    if any(x in t for x in ["burning urination", "pain urinating", "frequent urination", "uti"]):
        return "possible uti", "urinary symptoms"
    return "", ""

# ------------------------------
# Conversation state machine
# ------------------------------
REQUIRED_FIELDS = ["condition", "age", "sex", "pregnant", "allergies", "other_meds", "duration", "key_symptoms"]

if "messages" not in st.session_state:
    st.session_state.messages = []  # list of {role, content}
if "form" not in st.session_state:
    st.session_state.form = {f: None for f in REQUIRED_FIELDS}
if "finished" not in st.session_state:
    st.session_state.finished = False


def add_msg(role: str, content: str):
    st.session_state.messages.append({"role": role, "content": content})


def next_missing_field() -> str:
    for f in REQUIRED_FIELDS:
        if not st.session_state.form.get(f):
            return f
    return ""


def field_prompt(field: str) -> str:
    prompts = {
        "condition": "What condition or disease are you dealing with?.",
        "age": "How old are you? If for a child, specify the age in years (and months if <2 years).",
        "sex": "What's your sex assigned at birth? (male/female)",
        "pregnant": "Are you currently pregnant or breastfeeding? (yes/no/not applicable)",
        "allergies": "Any medication allergies? List them or say 'none'.",
        "other_meds": "Are you taking any other medicines or have chronic conditions (e.g., hypertension, diabetes)?",
        "duration": "How long have the symptoms been present? (e.g., 2 days)",
        "key_symptoms": "Briefly list your key symptoms (e.g., fever 38.5°C, sore throat, runny nose).",
    }
    return prompts.get(field, "Please provide more details.")


def incorporate_user_text(text: str):
    """Try to fill fields from free text and/or set the condition."""
    f = next_missing_field()
    if f == "condition":
        cond, alias = normalize_condition(text)
        if cond:
            st.session_state.form["condition"] = cond
        else:
            # keep free text as condition label if not matched
            st.session_state.form["condition"] = text.strip().lower()
    else:
        st.session_state.form[f] = text.strip()


def make_recommendation(data: Dict[str, Any]) -> str:
    cond = (data.get("condition") or "").lower()
    kb_key, _ = normalize_condition(cond)
    # if we couldn't map, treat as unknown
    if not kb_key:
        kb_section = {
            "overview": "Condition not recognized in demo list.",
            "otc": [
                "Consider symptomatic relief (e.g., paracetamol) if appropriate",
                "Hydration and rest"
            ],
            "seek_care": [
                "If symptoms are severe, persistent, or you are in a high-risk group"
            ],
            "notes": "Please consult a clinician/pharmacist for tailored advice."
        }
    else:
        kb_section = KB[kb_key]

    age_txt = data.get("age", "?")
    sex = (data.get("sex") or "").lower()
    pregnant = (data.get("pregnant") or "").lower()
    allergies = data.get("allergies", "none")
    other_meds = data.get("other_meds", "none")
    duration = data.get("duration", "")
    key_symptoms = data.get("key_symptoms", "")

    # Safety flags
    safety_flags = []
    ks = (key_symptoms or "").lower()
    red_flag_triggers = [
        ("shortness of breath", "Breathing difficulty"),
        ("chest pain", "Chest pain"),
        ("confusion", "Confusion"),
        ("black stool", "Black stools"),
        ("blood in stool", "Blood in stool"),
        ("vomiting blood", "Vomiting blood"),
        ("stiff neck", "Neck stiffness"),
        ("faint", "Fainting"),
        ("severe", "Severe symptoms"),
    ]
    for trig, label in red_flag_triggers:
        if trig in ks:
            safety_flags.append(label)

    # Pregnancy/child safeguards
    notes_extra = []
    if pregnant in ["yes", "pregnant", "breastfeeding", "breast feeding"]:
        notes_extra.append("Pregnant/breastfeeding: many meds require clinician approval; avoid NSAIDs unless advised.")
    try:
        age_val = int(str(age_txt).split()[0])
    except Exception:
        age_val = None
    if age_val is not None and age_val < 12:
        notes_extra.append("Child: dosing and medication choices differ — confirm with a clinician/pharmacist.")

    # Build response
    lines = []
    lines.append(f"**Condition (interpreted):** {kb_key or cond}")
    lines.append(f"**Duration:** {duration}")
    lines.append(f"**Key symptoms:** {key_symptoms}")
    lines.append("")

    lines.append("### What may help (OTC/supportive)")
    if kb_section.get("otc"):
        for item in kb_section["otc"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No OTC options recommended here. Seek clinician evaluation.")

    lines.append("")
    lines.append("### When to seek medical care urgently")
    for item in kb_section.get("seek_care", []):
        lines.append(f"- {item}")
    for flag in safety_flags:
        lines.append(f"- {flag}")

    if kb_section.get("notes") or notes_extra:
        lines.append("")
        lines.append("### Notes")
        if kb_section.get("notes"):
            lines.append(f"- {kb_section['notes']}")
        for n in notes_extra:
            lines.append(f"- {n}")

    lines.append("")
    lines.append(
        "> For Any Further Medication See A Doctor."
    )

    return "\n".join(lines)


# ------------------------------
# Chat UI
# ------------------------------
st.title("💊 PHARMACARE")
st.caption("Answer a Few questions and Get General OTC guidance + safety flags.")

# Show history
for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

# If conversation not finished, prompt next missing field
missing = next_missing_field()
if not st.session_state.finished:
    if missing:
        prompt = field_prompt(missing)
        with st.chat_message("assistant"):
            st.markdown(prompt)
        add_msg("assistant", prompt)
    else:
        # We have all fields, produce recommendation
        recommendation = make_recommendation(st.session_state.form)
        with st.chat_message("assistant"):
            st.markdown(recommendation)
        add_msg("assistant", recommendation)
        st.session_state.finished = True

# Chat input
user_text = st.chat_input("Type your answer here…")
if user_text:
    with st.chat_message("user"):
        st.markdown(user_text)
    add_msg("user", user_text)
    if not st.session_state.finished:
        incorporate_user_text(user_text)
        st.rerun()

# Controls
st.divider()
col1, col2, col3 = st.columns(3)
with col1:
    if st.button("🔁 Start Over"):
        st.session_state.messages = []
        st.session_state.form = {f: None for f in REQUIRED_FIELDS}
        st.session_state.finished = False
        st.rerun()
with col2:
    if st.button("💾 Download Conversation"):
        payload = {
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "form": st.session_state.form,
            "messages": st.session_state.messages,
        }
        st.download_button(
            label="Download JSON",
            data=json.dumps(payload, indent=2),
            file_name=f"pharmacy_chat_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.json",
            mime="application/json",
        )
with col3:
    st.markdown("\n")
    st.caption("Built with Streamlit.")

# ------------------------------
# End of app.py



Overwriting app.py


In [6]:
from pyngrok import ngrok

# Set the ngrok authtoken using the provided token
ngrok.set_auth_token("31NF03yQZ0tKLwGbTGsqnb6Ealw_4CqrFG7pZtHc4xt9AuGnR")

# Run Streamlit app
!streamlit run app.py &>/dev/null&

# Create an Ngrok tunnel for port 8501 (Streamlit default port)
public_url = ngrok.connect(8501, "http")

# Print the public URL to access the app
print(public_url)

NgrokTunnel: "https://f127b1827cd4.ngrok-free.app" -> "http://localhost:8501"
